# 2. Sistemas de Memoria Avanzados en LangChain

## Objetivos de Aprendizaje
- Comprender las limitaciones de la gestión manual de la memoria.
- Aprender a usar las clases de memoria de LangChain para automatizar el historial de chat.
- Implementar `ConversationBufferMemory` para un historial completo.
- Implementar `ConversationBufferWindowMemory` para limitar el tamaño del historial.
- Implementar `ConversationSummaryMemory` para resumir conversaciones y ahorrar tokens.
- Entender el puente hacia la planificación de agentes.

## El Problema con la Memoria Manual

En el notebook anterior, le dimos memoria a nuestro agente gestionando manualmente una lista `chat_history`. Aunque funcional, este enfoque tiene dos grandes inconvenientes:

1.  **Gestión Tediosa**: Actualizar la lista manualmente después de cada interacción es propenso a errores y no es escalable.
2.  **Límite de Contexto**: Enviar el historial completo en cada llamada consume tokens rápidamente. En conversaciones largas, esto puede exceder el límite de contexto del modelo, provocando errores o un alto costo.

LangChain resuelve esto con un sistema de **clases de Memoria** que automatizan el proceso y ofrecen estrategias para gestionar historiales largos.

### 1. Instalación y Configuración

In [1]:
!pip install langchain langchain-openai clangchain-classic openai wikipedia python-dotenv -q

ERROR: Could not find a version that satisfies the requirement clangchain-classic (from versions: none)

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\ingri\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for clangchain-classic


In [2]:
import os
import wikipedia
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

load_dotenv()  # Carga las variables de entorno desde el archivo .env

# Configuración del LLM (vía Groq)
try:
    llm = ChatOpenAI(
        model="openai/gpt-oss-120b",
        openai_api_base=os.environ.get("GROQ_BASE_URL"),
        openai_api_key=os.environ.get("GROQ_API_KEY"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado.


### 2. Agente y Herramientas (Sin Cambios)

Reutilizamos el mismo agente y herramientas. La innovación estará en cómo gestionamos la memoria.

In [3]:
from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client


@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]
client= Client()
# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
prompt = client.pull_prompt("hwchase17/openai-tools-agent", dangerously_pull_public_prompt=True)

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

✅ Agente y herramientas listos.


### 3. Automatizando con `ConversationBufferMemory`

Esta es la memoria más básica. Simplemente almacena todos los mensajes en una variable (el "buffer") y los pasa en cada llamada. Replica lo que hicimos manualmente, pero de forma automática.

In [ ]:
from langchain_classic.memory import ConversationBufferMemory

# memory_key debe coincidir con la variable de entrada en el prompt ('chat_history')
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

def chat_with_agent_buffer(query: str):
    # Carga el historial de la memoria
    history = memory.load_memory_variables({})["chat_history"]
    
    # Invoca al agente con el historial
    response = agent_executor.invoke({"input": query, "chat_history": history})
    
    # Guarda el nuevo par de pregunta/respuesta en la memoria
    memory.save_context({"input": query}, {"output": response["output"]})
    
    return response["output"] 

C:\Users\ingri\AppData\Local\Temp\ipykernel_17564\333789718.py:4: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [11]:
print("Primera pregunta...") 
response1 = chat_with_agent_buffer("Háblame del planeta Marte")
print(f"Respuesta 1: {response1}")

print("Segunda pregunta de seguimien...")
response2 = chat_with_agent_buffer("¿Tiene alguna luna?")
print(f"Respuesta 2: {response2}")

Primera pregunta...


> Entering new AgentExecutor chain...
### Marte: una visión completa del “planeta rojo”

| Aspecto | Detalle |
|---|---|
| **Posición en el Sistema Solar** | 4.º planeta desde el Sol, a una distancia media de **1,52 UA** (≈ 228  millones km). |
| **Tamaño y forma** | Diámetro ecuatorial **6 779 km** (≈ ½ del de la Tierra). Forma ligeramente oblata (aplanada en los polos) por su rotación. |
| **Masa y gravedad** | Masa **6,42 × 10²³ kg** (≈ 10,7 % de la masa terrestre). Gravedad superficial **3,71 m s⁻²** (≈ 38 % de la gravedad terrestre). |
| **Rotación y día** | Periodo de rotación **24 h 37 min** (un “sol”). Eje de inclinación **25,2°**, lo que produce estaciones similares a la Tierra, aunque más largas (≈ ½ año marciano = 334 soles). |
| **Órbita** | Excentricidad **0,0934** (ligeramente elíptica). Perihelio 206  millones km, afelio 249  millones km. Un año marciano dura **686,98 días terrestres** (≈ 669 soles). |
| **Atmósfera** | Muy tenue (presión media ≈ 6 

Como puedes ver, la función `chat_with_agent_buffer` ahora se encarga de cargar y guardar el historial, haciendo el proceso mucho más limpio.

### 4. Gestionando el Contexto con `ConversationBufferWindowMemory`

Para evitar que el historial crezca sin control, podemos usar `ConversationBufferWindowMemory`. Esta memoria solo conserva un número `k` de las últimas interacciones.

Esto es un buen equilibrio entre tener contexto y no sobrecargar al LLM.

In [16]:
from langchain_classic.memory import ConversationBufferWindowMemory

# Creamos una nueva memoria, esta vez con k=3 (solo recuerda los últimos 3 pares de mensajes)
memory_window = ConversationBufferWindowMemory(k=1, memory_key="chat_history", return_messages=True)

def chat_with_agent_window(query: str):
    history = memory_window.load_memory_variables({})["chat_history"]
    response = agent_executor.invoke({"input": query, "chat_history": history})
    memory_window.save_context({"input": query}, {"output": response["output"]})
    return response["output"]

In [17]:
print("Pregunta 1: ¿Quién fue Albert Einstein?")
chat_with_agent_window("¿Quién fue Albert Einstein?")

print("Pregunta 2: ¿Y Niels Bohr?")
chat_with_agent_window("¿Y Niels Bohr?")

print("Pregunta 3: ¿De qué trataba la primera pregunta que te hice?")
response3 = chat_with_agent_window("¿De qué trataba la primera pregunta que te hice?")
print(f"Respuesta 3: {response3}")

Pregunta 1: ¿Quién fue Albert Einstein?


> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'Albert Einstein'}`


Albert Einstein (pronunciación en alemán: /ˈalbɛʁt ˈaɪnʃtaɪn/ ();​ Ulm, Reino de Wurtemberg, 14 de marzo de 1879-Princeton, Nueva Jersey, 18 de abril de 1955) fue un físico alemán de origen judío, nacionalizado después suizo, austriaco y estadounidense, conocido principalmente por desarrollar la teoría de la relatividad. Se le considera el científico más importante, conocido y popular del siglo XX.​​
En 1905, cuando era un joven físico desconocido, empleado en la Oficina de Patentes de Berna, publicó su teoría de la relatividad especial.Albert Einstein (14 de marzo de 1879 – 18 de abril de 1955) fue un físico de origen judío nacido en Ulm, Alemania, que se convirtió en una de las figuras científicas más influyentes del siglo XX. Es célebre por desarrollar la teoría de la relatividad (especial y general), que revolucionó la comprensión 

El agente probablemente no podrá responder a la tercera pregunta, porque con `k=1`, la interacción sobre Einstein ya fue descartada de la memoria para dar paso a la de Niels Bohr.

### 5. Ahorrando Tokens con `ConversationSummaryMemory`

Esta es la estrategia más avanzada. En lugar de guardar los mensajes completos, utiliza un LLM para crear un resumen de la conversación a medida que avanza. En cada nueva interacción, el resumen se actualiza y se pasa como contexto.

Es ideal para conversaciones muy largas donde los detalles específicos se vuelven menos importantes que el contexto general.

In [18]:
from langchain_classic.memory import ConversationSummaryMemory

# Esta memoria necesita un LLM para hacer los resúmenes
memory_summary = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)

def chat_with_agent_summary(query: str):
    history = memory_summary.load_memory_variables({})["chat_history"]
    response = agent_executor.invoke({"input": query, "chat_history": history})
    memory_summary.save_context({"input": query}, {"output": response["output"]})
    return response["output"]

In [19]:
print("Pregunta 1: Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?")
chat_with_agent_summary("Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?")

print("Pregunta 2: Suena bien. ¿Qué ciudades me recomiendas visitar en un primer viaje?")
chat_with_agent_summary("¿Qué ciudades me recomiendas visitar en un primer viaje?")

print("Pregunta 3: ¿Sobre qué estábamos hablando?")
response3_summary = chat_with_agent_summary("¿Sobre qué estábamos hablando?")
print(f"Respuesta 3: {response3_summary}")

print("Pregunta 4: que pasatiempos se pueden hacer en el viaje")
response3_summary = chat_with_agent_summary("que pasatiempos se pueden hacer en el viaje")
print(f"Respuesta 4: {response3_summary}")

print("Pregunta 5: por que otro pais podria organizar mi siguiente viaje")
response3_summary = chat_with_agent_summary("por que otro pais podria organizar mi siguiente viaje")
print(f"Respuesta 5: {response3_summary}")

print("Pregunta 6: ¿Sobre qué estábamos hablando?")
response3_summary = chat_with_agent_summary("¿Sobre qué estábamos hablando?")
print(f"Respuesta 6: {response3_summary}")

Pregunta 1: Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?


> Entering new AgentExecutor chain...
¡Claro! Japón tiene una gran variedad de climas y eventos a lo largo del año, por lo que la “mejor” época depende de lo que más te interese (paisajes, festivales, precios, etc.). Aquí tienes un resumen de las cuatro estaciones y sus principales atractivos para que puedas decidir:

| Época | Clima | Ventajas | Desventajas |
|-------|-------|----------|-------------|
| **Primavera (marzo‑mayo)** | Temperaturas suaves (10‑20 °C). La humedad aún es baja. | - **Sakura (cerezos en flor)**: el espectáculo más icónico, con picos en Tokio, Kioto y Osaka a finales de marzo‑principios de abril.<br>- Festivales como el **Hanami** (picnics bajo los cerezos) y el **Golden Week** (aunque muy concurrido).<br>- Paisajes verdes y flores de ciruelo. | - Gran afluencia de turistas, lo que eleva precios de alojamiento y vuelos.<br>- En la última semana de abril y primera de mayo (Golden

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0te8fmmesdbh9f59tngyz84` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 3489, Requested 5482. Please try again in 7.2825s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

El agente debería poder responder correctamente, ya que la memoria no contiene los mensajes literales, sino un resumen de que el usuario está planeando un viaje a Japón.

## Conclusiones y Transición al Módulo IL2.3

Hemos visto cómo las clases de memoria de LangChain nos permiten crear agentes conversacionales mucho más robustos y eficientes, superando los desafíos de la gestión manual y el tamaño del contexto.

**Resumen de Estrategias de Memoria:**
- **`ConversationBufferMemory`**: Simple y completa. Ideal para conversaciones cortas donde cada detalle importa.
- **`ConversationBufferWindowMemory`**: Eficiente y balanceada. Perfecta para chatbots de servicio al cliente que necesitan contexto reciente.
- **`ConversationSummaryMemory`**: Ahorradora de tokens y escalable. La mejor opción para asistentes de largo plazo y análisis de conversaciones extensas.

Con un agente que puede recordar de manera efectiva, hemos completado una pieza fundamental del rompecabezas. Sin embargo, las conversaciones del mundo real a menudo implican tareas que no se resuelven con una sola herramienta o en un solo paso.

**¿Qué pasa si el agente necesita buscar información, luego calcular algo con esa información y finalmente enviar un correo electrónico?**

Esto requiere que el agente pueda **planificar** una secuencia de acciones. Debe ser capaz de descomponer un objetivo complejo en pasos más pequeños y ejecutarlos en el orden correcto.

Esta es la transición perfecta al **Módulo IL2.3: Planificación y Orquestación de Agentes**, donde exploraremos cómo los agentes pueden crear, seguir y adaptar planes para resolver problemas complejos y multi-paso.

In [20]:
print("Pregunta 1: Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?")
chat_with_agent_buffer("Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?")

print("Pregunta 2: Suena bien. ¿Qué ciudades me recomiendas visitar en un primer viaje?")
chat_with_agent_buffer("¿Qué ciudades me recomiendas visitar en un primer viaje?")

print("Pregunta 3: ¿Sobre qué estábamos hablando?")
response3_summary = chat_with_agent_buffer("¿Sobre qué estábamos hablando?")
print(f"Respuesta 3: {response3_summary}")

print("Pregunta 4: que pasatiempos se pueden hacer en el viaje")
response3_summary = chat_with_agent_buffer("que pasatiempos se pueden hacer en el viaje")
print(f"Respuesta 4: {response3_summary}")

print("Pregunta 5: por que otro pais podria organizar mi siguiente viaje")
response3_summary = chat_with_agent_buffer("por que otro pais podria organizar mi siguiente viaje")
print(f"Respuesta 5: {response3_summary}")

print("Pregunta 6: ¿Sobre qué estábamos hablando?")
response3_summary = chat_with_agent_buffer("¿Sobre qué estábamos hablando?")
print(f"Respuesta 6: {response3_summary}")

Pregunta 1: Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?


> Entering new AgentExecutor chain...
## ¿Cuál es la mejor época para viajar a Japón?

### 1. Resumen rápido
| Época | Clima | Atractivos principales | Multitud / precios |
|-------|-------|------------------------|--------------------|
| **Marzo‑abril (primavera)** | Templado, días soleados; lluvias escasas | **Sakura** (cerezos en flor) en todo el país; festivales de flores; clima ideal para caminar | Alta demanda, precios de alojamiento y vuelos suben; reservas con antelación son clave |
| **Octubre‑noviembre (otoño)** | Fresco‑templado, poca lluvia | **Koyo** (colores otoñales) en parques y templos; clima cómodo para turismo urbano y rural | Similar a la primavera en popularidad, pero ligeramente menos saturado que la semana de los cerezos |
| **Mayo‑junio (principios de verano)** | Cálido, humedad moderada | Flores de **wisteria**, **iris**, y **hydrangea**; menos turistas que la primavera | Precio

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01m0te8fmmesdbh9f59tngyz84` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 9734, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
def primera_funcion_2(query: str):

    return chat_with_agent_buffer(query)

query = input("Ingrese su pregunta: ")

print(primera_funcion_2(query))



> Entering new AgentExecutor chain...
¡Hola! 👋  

### ¿Qué es la IA (Inteligencia Artificial)?

La **Inteligencia Artificial (IA)** es un conjunto de técnicas y sistemas que permiten a las máquinas **“pensar”, aprender y tomar decisiones** de forma similar (aunque mucho más limitada) a como lo hacen los seres humanos. En lugar de seguir instrucciones fijas paso a paso, una IA puede:

1. **Reconocer patrones** en datos (por ejemplo, identificar caras en fotos o detectar fraudes en transacciones).  
2. **Aprender de la experiencia** mediante algoritmos de *machine learning* (aprendizaje automático): cuanto más datos recibe, mejor se vuelve en la tarea que se le ha asignado.  
3. **Adaptarse y generalizar** a situaciones nuevas sin necesidad de reprogramarla completamente.  

### Tipos principales de IA

| Tipo | Descripción | Ejemplos cotidianos |
|------|-------------|---------------------|
| **IA estrecha (o débil)** | Diseñada para una tarea específica (reconocimiento de voz, recome